# LegalEagle Benchmark Eval Suite

Compare three tracks: zero-shot OpenAI baseline, RAG-only agent, and LegalBERT + RAG + OpenAI agent.

In [ ]:
!pip install -q pandas scikit-learn pydantic openai


In [ ]:
import pandas as pd
from sklearn.metrics import precision_recall_fscore_support

test_df = pd.read_json('/content/legaleagle_test.jsonl', lines=True)
test_df.head()


In [ ]:
def macro_metrics(y_true, y_pred):
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    return {'precision_macro': precision, 'recall_macro': recall, 'f1_macro': f1}

def hit_at_k(expected, ranked, k=5):
    return float(expected in ranked[:k])

def schema_validity(rows):
    required = {'contract_summary', 'overall_risk_score', 'risk_level', 'findings', 'missing_clauses', 'reviewer_notes'}
    return sum(required.issubset(set(row.keys())) for row in rows) / max(1, len(rows))


In [ ]:
# Fill these arrays with predictions exported from each pipeline.
zero_shot_predictions = []
rag_only_predictions = []
legalbert_rag_predictions = []

results = []
for name, preds in [
    ('zero_shot_openai', zero_shot_predictions),
    ('rag_only_agent', rag_only_predictions),
    ('legalbert_rag_openai_agent', legalbert_rag_predictions),
]:
    if len(preds) == len(test_df):
        metrics = macro_metrics(test_df['label'], preds)
        metrics['pipeline'] = name
        results.append(metrics)

pd.DataFrame(results)


## Portfolio Reporting

Report the final benchmark as:

- clause classification precision / recall / F1
- retrieval hit@3 and hit@5
- structured output validity
- end-to-end finding recall on held-out contracts
- qualitative examples where memory feedback corrected later reviews